## Canadian Monthly Retail Trade Sales Dataset — Cleaning & Feature Engineering

## About the Dataset

**Source:** Statistics Canada — [Table 20-10-0056-01: Monthly Retail Trade Sales by Province and Territory](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=2010005601)  
**Publisher:** Statistics Canada (Government of Canada)  
**Frequency:** Monthly (starting January 2017)  
**Coverage:** Canada, provinces, territories, and Census Metropolitan Areas (CMAs)  
**Classification:** North American Industry Classification System (NAICS)  
**Values unit:** Dollars × 1,000 (`SCALAR_FACTOR = thousands`)  
**Last updated:** February 20, 2026  
**Direct CSV download:** [20100056-eng.zip](https://www150.statcan.gc.ca/n1/tbl/csv/20100056-eng.zip)

This dataset captures **monthly retail trade sales** across Canadian geographies and NAICS-based industry subsectors. Both **unadjusted** and **seasonally adjusted** figures are included, enabling trend and seasonal analysis.

## Use Cases

- **Sales trend analysis** – track retail revenue over time by industry and region  
- **Seasonal decomposition** – compare unadjusted vs. seasonally adjusted figures  
- **Regional performance** – benchmark provinces, territories, and CMAs against national totals  
- **Industry benchmarking** – compare subsectors (e.g., grocery vs. auto vs. e-commerce)  
- **E-commerce growth** – monitor the rise of online retail relative to total retail sales  

## Raw Dataset Columns (17 original columns from Statistics Canada)

| # | Column | Type | Description |
|---|--------|------|-------------|
| 1 | `REF_DATE` | string → datetime | Reference month in `YYYY-MM` format (e.g., `2017-01`) |
| 2 | `GEO` | string | Geographic area name (e.g., "Canada", "Ontario", "Toronto, Ontario") |
| 3 | `DGUID` | string | Dissemination Geography Unique Identifier — internal StatCan geo code (**dropped**) |
| 4 | `North American Industry Classification System (NAICS)` | string | NAICS industry name with code in brackets (e.g., `Retail trade [44-45]`) |
| 5 | `Sales` | string | Sales type: `Total retail sales` or `Retail e-commerce sales` |
| 6 | `Adjustments` | string | Seasonal treatment: `Unadjusted` or `Seasonally adjusted` |
| 7 | `UOM` | string | Unit of measure — always `Dollars` (**dropped** after use) |
| 8 | `UOM_ID` | integer | Numeric code for UOM — always `81` (**dropped**) |
| 9 | `SCALAR_FACTOR` | string | Scale applied to VALUE — always `thousands` (**dropped** after scaling) |
| 10 | `SCALAR_ID` | integer | Numeric code for scalar — always `3` (**dropped**) |
| 11 | `VECTOR` | string | StatCan internal time-series vector ID (e.g., `v1446859481`) (**dropped**) |
| 12 | `COORDINATE` | string | StatCan internal coordinate reference (e.g., `1.1.1.1`) (**dropped**) |
| 13 | `VALUE` | float → int | Sales figure in thousands of dollars as published by StatCan |
| 14 | `STATUS` | string | Data quality flag — empty means no issues (**dropped**) |
| 15 | `SYMBOL` | string | Special symbol for revised/preliminary values — mostly empty (**dropped**) |
| 16 | `TERMINATED` | string | Flags discontinued series — mostly empty (**dropped**) |
| 17 | `DECIMALS` | integer | Number of decimal places — always `0` (**dropped**) |

## Engineered Columns (added during cleaning & feature engineering)

| # | Column | Description |
|---|--------|-------------|
| 1 | `Year` | Calendar year extracted from `REF_DATE` |
| 2 | `Month` | Calendar month number (1–12) extracted from `REF_DATE` |
| 3 | `Month_Name` | Full month name (e.g., January) extracted from `REF_DATE` |
| 4 | `Quarter` | Fiscal quarter (1–4) extracted from `REF_DATE` |
| 5 | `Industry` | NAICS name with codes stripped (e.g., `Retail trade [44-45]` → `Retail trade`) |
| 6 | `Geo_Level` | Geographic tier: `National` / `Province` / `Territory` / `City` |
| 7 | `Sales_Actual` | `VALUE × 1,000` — sales expressed in actual dollar amounts |

In [242]:
# Import essential libraries
# pandas for data manipulation, numpy for numerical operations
# Set float display format to avoid scientific notation
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:,.0f}'.format)

In [243]:
# Load the Canadian retail dataset from CSV
# Display the first 5 rows to get a quick overview of the data
df = pd.read_csv("Dataset/20100056.csv")
df.head()

,REF_DATE,GEO,DGUID,North American Industry Classification System (NAICS),Sales,Adjustments,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2017-01,Canada,2021A000011124,Retail trade [44-45],Total retail sales,Unadjusted,Dollars,81,thousands,3,v1446859481,1.1.1.1,"41,377,009",NaN,NaN,NaN,0
1,2017-01,Canada,2021A000011124,Retail trade [44-45],Total retail sales,Seasonally adjusted,Dollars,81,thousands,3,v1446859483,1.1.1.2,"50,417,235",NaN,NaN,NaN,0
2,2017-01,Canada,2021A000011124,Retail trade [44-45],Retail e-commerce sales,Unadjusted,Dollars,81,thousands,3,v1446859482,1.1.2.1,"1,110,045",NaN,NaN,NaN,0
3,2017-01,Canada,2021A000011124,Retail trade [44-45],Retail e-commerce sales,Seasonally adjusted,Dollars,81,thousands,3,v1446859484,1.1.2.2,"1,236,885",NaN,NaN,NaN,0
4,2017-01,Canada,2021A000011124,Motor vehicle and parts dealers [441],Total retail sales,Unadjusted,Dollars,81,thousands,3,v1446859485,1.2.1.1,"10,162,441",NaN,NaN,NaN,0


In [244]:
# Inspect data types, column names, non-null counts, and memory usage
# Helps identify columns with missing values and incorrect dtypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76482 entries, 0 to 76481
Data columns (total 17 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   REF_DATE                                               76482 non-null  object 
 1   GEO                                                    76482 non-null  object 
 2   DGUID                                                  76482 non-null  object 
 3   North American Industry Classification System (NAICS)  76482 non-null  object 
 4   Sales                                                  76482 non-null  object 
 5   Adjustments                                            76482 non-null  object 
 6   UOM                                                    76482 non-null  object 
 7   UOM_ID                                                 76482 non-null  int64  
 8   SCALAR_FACTOR                                 

In [245]:
# Check the dimensions of the dataset: (rows, columns)
# Confirms the expected 34,500 records and 17 features
df.shape

(76482, 17)

In [246]:
# Count missing (NaN) values per column
# Identifies which columns need imputation or dropping
df.isnull().sum()

REF_DATE                                                     0
GEO                                                          0
DGUID                                                        0
North American Industry Classification System (NAICS)        0
Sales                                                        0
Adjustments                                                  0
UOM                                                          0
UOM_ID                                                       0
SCALAR_FACTOR                                                0
SCALAR_ID                                                    0
VECTOR                                                       0
COORDINATE                                                   0
VALUE                                                     9988
STATUS                                                   39321
SYMBOL                                                   76482
TERMINATED                                             

In [247]:
# Count fully duplicated rows across all columns
# Duplicate records can distort analysis and model training
df.duplicated().sum()

0

In [248]:
# Select all object (string/text) columns for whitespace inspection
# Text columns are the most common source of formatting issues
text_cols = df.select_dtypes(include='object').columns

In [249]:
# Check each text column for leading or trailing whitespace
# Unstripped spaces cause silent mismatches in groupby and merges
for col in text_cols:
    print(col, df[col].str.contains(r'^\s|\s$', na=False).sum())

REF_DATE 0
GEO 0
DGUID 0
North American Industry Classification System (NAICS) 0
Sales 0
Adjustments 0
UOM 0
SCALAR_FACTOR 0
VECTOR 0
COORDINATE 0
STATUS 0


In [250]:
# Convert the REF_DATE column from string to datetime format
# Required before any date-based feature extraction can be done
df['REF_DATE'] = pd.to_datetime(df['REF_DATE'])

In [251]:
# Extract time-based features from the parsed date column
# Year, Month, Month_Name, and Quarter enable temporal analysis
df['Year'] = df['REF_DATE'].dt.year
df['Month'] = df['REF_DATE'].dt.month
df['Month_Name'] = df['REF_DATE'].dt.month_name()
df['Quarter'] = df['REF_DATE'].dt.quarter

In [252]:
# Preview the first 5 rows after adding the new date features
# Confirms the new columns (Year, Month, Month_Name, Quarter) were created correctly
df.head()

,REF_DATE,GEO,DGUID,North American Industry Classification System (NAICS),Sales,Adjustments,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,...,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS,Year,Month,Month_Name,Quarter
0,2017-01-01,Canada,2021A000011124,Retail trade [44-45],Total retail sales,Unadjusted,Dollars,81,thousands,3,...,1.1.1.1,"41,377,009",NaN,NaN,NaN,0,2017,1,January,1
1,2017-01-01,Canada,2021A000011124,Retail trade [44-45],Total retail sales,Seasonally adjusted,Dollars,81,thousands,3,...,1.1.1.2,"50,417,235",NaN,NaN,NaN,0,2017,1,January,1
2,2017-01-01,Canada,2021A000011124,Retail trade [44-45],Retail e-commerce sales,Unadjusted,Dollars,81,thousands,3,...,1.1.2.1,"1,110,045",NaN,NaN,NaN,0,2017,1,January,1
3,2017-01-01,Canada,2021A000011124,Retail trade [44-45],Retail e-commerce sales,Seasonally adjusted,Dollars,81,thousands,3,...,1.1.2.2,"1,236,885",NaN,NaN,NaN,0,2017,1,January,1
4,2017-01-01,Canada,2021A000011124,Motor vehicle and parts dealers [441],Total retail sales,Unadjusted,Dollars,81,thousands,3,...,1.2.1.1,"10,162,441",NaN,NaN,NaN,0,2017,1,January,1


In [253]:
# View all unique geographic values in the GEO column
# Needed to plan the geo-level classification logic below
df['GEO'].unique()

array(['Canada', 'Newfoundland and Labrador', 'Prince Edward Island',
       'Nova Scotia', 'New Brunswick', 'Quebec', 'Montréal, Quebec',
       'Ontario', 'Toronto, Ontario', 'Manitoba', 'Saskatchewan',
       'Alberta', 'British Columbia', 'Vancouver, British Columbia',
       'Yukon', 'Northwest Territories', 'Nunavut', 'Quebec, Quebec',
       'Gatineau, Quebec', 'Ottawa, Ontario', 'Winnipeg, Manitoba',
       'Calgary, Alberta', 'Edmonton, Alberta'], dtype=object)

In [254]:
# Define reference lists for each geographic level
# Used to map raw GEO strings to a structured Geo_Level category
country = ["Canada"]

territories = [
    "Yukon",
    "Northwest Territories",
    "Nunavut"
]

provinces = [
    "Newfoundland and Labrador",
    "Prince Edward Island",
    "Nova Scotia",
    "New Brunswick",
    "Quebec",
    "Ontario",
    "Manitoba",
    "Saskatchewan",
    "Alberta",
    "British Columbia"
]

cities = [
    "Montréal, Quebec",
    "Toronto, Ontario",
    "Vancouver, British Columbia",
    "Quebec, Quebec",
    "Gatineau, Quebec",
    "Ottawa, Ontario",
    "Winnipeg, Manitoba",
    "Calgary, Alberta",
    "Edmonton, Alberta"
]

In [255]:
# Apply the geo_level function to classify each row
# Creates a new 'Geo_Level' column: National / Province / Territory / City / Unknown
def geo_level(x):
    if x in country:
        return "National"
    elif x in territories:
        return "Territory"
    elif x in provinces:
        return "Province"
    elif x in cities:
        return "City"
    else:
        return "Unknown"

df['Geo_Level'] = df['GEO'].apply(geo_level)

In [256]:
# Count the number of records at each geographic level
# Validates the classification — helps catch unmatched or misspelled region names
df['Geo_Level'].value_counts()

Geo_Level
Province     33890
City         25647
Territory    10167
National      6778
Name: count, dtype: int64

In [257]:
# Split the dataframe into separate subsets by geographic level
# Enables level-specific analysis (e.g., provincial vs. city trends)
country_df = df[df['Geo_Level'] == "Country"]
province_df = df[df['Geo_Level'] == "Province"]
city_df = df[df['Geo_Level'] == "City"]
territory_df = df[df['Geo_Level'] == "Territory"]

In [260]:
# Rename the long NAICS column to the shorter 'Industry' label
# Improves readability in all subsequent operations
df.rename(columns={
    'North American Industry Classification System (NAICS)': 'Industry'
}, inplace=True)

In [262]:
# Strip NAICS classification codes (e.g., [45111]) from industry names
# Leaves only the plain-text industry description
df['Industry'] = df['Industry'].str.replace(r'\s*\[.*?\]', '', regex=True)

In [263]:
# Inspect all unique industry names after cleaning
# Confirms the regex removal worked and no codes remain
df['Industry'].unique()

array(['Retail trade', 'Motor vehicle and parts dealers',
       'Automobile dealers', 'New car dealers', 'Used car dealers',
       'Other motor vehicle dealers',
       'Automotive parts, accessories and tire retailers',
       'Building material and garden equipment and supplies dealers',
       'Food and beverage retailers', 'Grocery and convenience retailers',
       'Supermarkets and other grocery retailers (except convenience retailers)',
       'Convenience retailers and vending machine operators',
       'Specialty food retailers', 'Beer, wine and liquor retailers',
       'Furniture, home furnishings, electronics and appliances retailers',
       'Furniture, floor covering, window treatment and other home furnishings retailers',
       'Furniture retailers',
       'Floor covering, window treatment and other home furnishing retailers',
       'Electronics and appliances retailers',
       'General merchandise retailers',
       'Health and personal care retailers',
       'Ga

In [264]:
# Explore unique values in the Sales column
# Understand what sale types or categories are present in the data
df['Sales'].value_counts()

Sales
Total retail sales         76262
Retail e-commerce sales      220
Name: count, dtype: int64

In [265]:
# Explore unique values in the Adjustments column
# Understand what adjustment types exist (e.g., seasonal, unadjusted)
df['Adjustments'].value_counts()

Adjustments
Unadjusted             71333
Seasonally adjusted     5149
Name: count, dtype: int64

In [266]:
# Drop administrative/metadata columns that add no analytical value
# Reduces noise and memory usage in the cleaned dataframe
df.drop(columns=[
    'DGUID',
    'UOM_ID',
    'SCALAR_ID',
    'VECTOR',
    'COORDINATE',
    'STATUS',
    'SYMBOL',
    'TERMINATED',
    'DECIMALS'
], inplace=True)

In [267]:
# Identify rows where VALUE (sales figure) is missing
# Inspecting context (GEO, Industry) helps decide whether to impute or drop
df[df['VALUE'].isna()][['GEO', 'Industry', 'Sales', 'Adjustments']]

,GEO,Industry,Sales,Adjustments
68,Newfoundland and Labrador,Building material and garden equipment and sup...,Total retail sales,Unadjusted
73,Newfoundland and Labrador,Specialty food retailers,Total retail sales,Unadjusted
74,Newfoundland and Labrador,"Beer, wine and liquor retailers",Total retail sales,Unadjusted
75,Newfoundland and Labrador,"Furniture, home furnishings, electronics and a...",Total retail sales,Unadjusted
76,Newfoundland and Labrador,"Furniture, floor covering, window treatment an...",Total retail sales,Unadjusted
...,...,...,...,...
76477,Nunavut,"Jewellery, luggage and leather goods retailers",Total retail sales,Unadjusted
76478,Nunavut,"Sporting goods, hobby, musical instrument, boo...",Total retail sales,Unadjusted
76479,Nunavut,"Sporting goods, hobby, musical instrument, boo...",Total retail sales,Unadjusted
76480,Nunavut,Miscellaneous retailers,Total retail sales,Unadjusted


In [268]:
# Drop all rows where VALUE is NaN
# Rows without a sales figure cannot contribute to any quantitative analysis
df = df.dropna(subset=['VALUE'])

In [269]:
# Check units of measure present in the UOM column
# Ensures all records use a consistent unit before scaling
df['UOM'].value_counts()

UOM
Dollars    66494
Name: count, dtype: int64

In [270]:
# Check the scalar factor applied to VALUE
# Confirms all values use the same multiplier (e.g., thousands) before conversion
df['SCALAR_FACTOR'].value_counts()

SCALAR_FACTOR
thousands    66494
Name: count, dtype: int64

In [271]:
# Cast VALUE from float to integer
# Sales figures are whole numbers; int64 is more memory-efficient and semantically correct
df['VALUE'] = df['VALUE'].astype('int64')

C:\Users\Hamza\AppData\Local\Temp\ipykernel_24112\4218774865.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['VALUE'] = df['VALUE'].astype('int64')


In [272]:
# Create Sales_Actual by multiplying VALUE by 1,000
# Reverses the scalar factor to express sales in real dollar amounts
df = df.copy()
df['Sales_Actual'] = df['VALUE'] * 1000

In [273]:
# Remove SCALAR_FACTOR and UOM — now redundant after scaling
# Keeps the dataframe clean and free of helper columns
df.drop(columns=['SCALAR_FACTOR', 'UOM'], inplace=True)

In [274]:
# Reorder columns into a logical, readable sequence
# Date fields → geography → industry → sales metrics
df = df[
    [
        'REF_DATE',
        'Year',
        'Month',
        'Month_Name',
        'Quarter',
        'GEO',
        'Geo_Level',
        'Industry',
        'Sales',
        'Adjustments',
        'VALUE',
        'Sales_Actual'
    ]
]

In [275]:
# Final schema check: dtypes, non-null counts, and memory
# Validates the cleaned and engineered dataframe before export
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 66494 entries, 0 to 76473
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   REF_DATE      66494 non-null  datetime64[ns]
 1   Year          66494 non-null  int32         
 2   Month         66494 non-null  int32         
 3   Month_Name    66494 non-null  object        
 4   Quarter       66494 non-null  int32         
 5   GEO           66494 non-null  object        
 6   Geo_Level     66494 non-null  object        
 7   Industry      66494 non-null  object        
 8   Sales         66494 non-null  object        
 9   Adjustments   66494 non-null  object        
 10  VALUE         66494 non-null  int64         
 11  Sales_Actual  66494 non-null  int64         
dtypes: datetime64[ns](1), int32(3), int64(2), object(6)
memory usage: 5.8+ MB


In [276]:
# Display the complete cleaned dataframe
# Final visual review before writing to disk
df

,REF_DATE,Year,Month,Month_Name,Quarter,GEO,Geo_Level,Industry,Sales,Adjustments,VALUE,Sales_Actual
0,2017-01-01,2017,1,January,1,Canada,National,Retail trade,Total retail sales,Unadjusted,41377009,41377009000
1,2017-01-01,2017,1,January,1,Canada,National,Retail trade,Total retail sales,Seasonally adjusted,50417235,50417235000
2,2017-01-01,2017,1,January,1,Canada,National,Retail trade,Retail e-commerce sales,Unadjusted,1110045,1110045000
3,2017-01-01,2017,1,January,1,Canada,National,Retail trade,Retail e-commerce sales,Seasonally adjusted,1236885,1236885000
4,2017-01-01,2017,1,January,1,Canada,National,Motor vehicle and parts dealers,Total retail sales,Unadjusted,10162441,10162441000
...,...,...,...,...,...,...,...,...,...,...,...,...
76451,2026-02-01,2026,2,February,1,Nunavut,Territory,Retail trade,Total retail sales,Unadjusted,60983,60983000
76452,2026-02-01,2026,2,February,1,Nunavut,Territory,Retail trade,Total retail sales,Seasonally adjusted,66714,66714000
76460,2026-02-01,2026,2,February,1,Nunavut,Territory,Food and beverage retailers,Total retail sales,Unadjusted,41580,41580000
76462,2026-02-01,2026,2,February,1,Nunavut,Territory,Supermarkets and other grocery retailers (exce...,Total retail sales,Unadjusted,39073,39073000


In [277]:
# Generate descriptive statistics for all numeric columns
# Provides a quick sanity check on value ranges and distributions
df.describe()

,REF_DATE,Year,Month,Quarter,VALUE,Sales_Actual
count,66494,"66,494","66,494","66,494","66,494","66,494"
mean,2021-10-30 05:18:55.753602304,"2,021",6,2,"1,198,721","1,198,721,394"
min,2017-01-01 00:00:00,"2,017",1,1,1,"1,000"
25%,2019-09-01 00:00:00,"2,019",3,1,"37,812","37,811,750"
50%,2021-12-01 00:00:00,"2,021",6,2,"164,909","164,909,000"
75%,2024-01-01 00:00:00,"2,024",9,3,"676,897","676,896,750"
max,2026-02-01 00:00:00,"2,026",12,4,"76,831,848","76,831,848,000"
std,NaN,3,3,1,"4,291,667","4,291,666,946"


In [279]:
# Export the cleaned and feature-engineered dataset to CSV
# Saved as retail_sales_cleaned.csv for downstream analysis or modelling
df.to_csv("Dataset/retail_sales_cleaned.csv", index=False)